# Building a RAG System with LangChain

## Author: Jorge Soto Tripiana

# Experimental Project Objectives: LLM with RAG on PDF Documents

**General Objective:**
Evaluate the impact of RAG in LLM responses and compare the effect different hyperparameters configurations have on the accuracy of responses generated, using a set of documents as knowledge base.


**Specifically:**

1. **RAG vs. No RAG:**
  Demonstrate the improvement in accuracy and coherence while reducing hallucinations when using RAG, compared to a traditional Zero-Shot approach.


2. **Chunking Analysis:**
  Observe how the sizes of chunks extracted from the documents affects the quality of the output.


3. **Parameter *k* Study (Retrieval):**
   Determine the optimal number of retrieved chunks (*k* = 1, 3, 5) to maximize response quality, balancing the amount of context and avoiding noise.


4. **Model Comparison:**
   Compare the performance of different embedding models towards retrieval.


5. **Evaluation:**
   Implement automatic NLP evaluation metrics (ROUGE, BLEU for generation; Hit Rate for the retriever) to determine the performance of each configuration.

In [ ]:
# setting up dependencies

!pip install langchain-docling
!pip install bitsandbytes # 4bit quant
!pip install -q flash-linear-attention # attention optimizations

In [ ]:
# libraries

import torch
import os
from google.colab import drive
from langchain_docling.loader import DoclingLoader
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer

In [ ]:
# This cell stores the weights in drive,avoiding redundant downloads.

drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen3.5-9B"
SAVE_DIR = "/content/drive/MyDrive/Models/Qwen3.5-9B-4bit"

if os.path.exists(f"{SAVE_DIR}/config.json"):
    # load model
    pipe = pipeline(
        task="text-generation",
        model=SAVE_DIR,
        dtype=torch.bfloat16,
        device_map="auto",
    )
else:
    # download & store in drive
    pipe = pipeline(
        task="text-generation",
        model=MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
        model_kwargs={"quantization_config": BitsAndBytesConfig(
            # less vram usage & faster compute (bfloat16)
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)},
    )

    pipe.model.save_pretrained(SAVE_DIR)
    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(SAVE_DIR)

In [ ]:
# PDF loader
FILE_PATH = "document.pdf"
loader = DoclingLoader(file_path=FILE_PATH)

In [ ]:
# load
docs = loader.load()

In [ ]:
# LLM test

messages = [
    {"role" : "user", "content" : "What's the use case for Retrieval Augmented Generation sytems?"},
]

out = pipe(messages)
print(out[0]["generated_text"][-1]["content"])


